アップサンプリング

In [16]:
import pandas as pd
import os

# --- 設定 ---
pre = "/home/hiromu/energy"
INPUT_CSV_PATH = f'{pre}/data/26_0413_moreclean/feature_huall.csv'
IMG_ROOT = f'{pre}/data/26_0413_moreclean'
INPUT_CSV_PATH = f'{pre}/data/1201_humomentstest/feature_huall.csv'
# IMG_ROOT = f'{pre}/data/1201_humomentstest'
OUTPUT_DIR = f'{pre}/src/FF/AFF/dataset_upsamp_moreclean' # 保存先

TRAIN_SIZES = [10, 50, 100] # 任意の比較サイズ

# ★ 評価データの枚数設定（全クラスの合計枚数）
FIXED_TEST_TOTAL = 144
FIXED_VAL_TOTAL = 144

# ★ 学習データのアップサンプリング（重複抽出）を許可するかのフラグ
ALLOW_UPSAMPLING = True

RANDOM_STATE = 42

def print_distribution(df, name):
    if df.empty:
        print(f"[{name}] データがありません。")
        return
    dist = df['Label'].value_counts().sort_index().to_dict()
    dist_str = ", ".join([f"{k}: {v}枚" for k, v in dist.items()])
    print(f"[{name}] 合計: {len(df)}枚 (内訳 -> {dist_str})")


def create_comparison_datasets():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # 1. データ読み込みと有効データ抽出
    print("Loading data...")
    raw_df = pd.read_csv(INPUT_CSV_PATH)
    valid_indices = []
    for i in range(len(raw_df)):
        img_path = os.path.join(IMG_ROOT, 'mask', raw_df.iloc[i]['category'], raw_df.iloc[i]['filename'])
        if os.path.exists(img_path):
            valid_indices.append(i)
    valid_df = raw_df.iloc[valid_indices].reset_index(drop=True)
    
    print("\n--- 元データの分布 ---")
    print_distribution(valid_df, "Original Valid Data")
    print("-" * 30 + "\n")

    # 2. Test / Val / Train の分割（枚数固定・クラス均等）
    num_classes = valid_df['Label'].nunique()
    test_per_class = FIXED_TEST_TOTAL // num_classes
    val_per_class = FIXED_VAL_TOTAL // num_classes
    
    test_fixed = pd.DataFrame()
    val_fixed = pd.DataFrame()
    train_master = pd.DataFrame()

    for label in valid_df['Label'].unique():
        subset = valid_df[valid_df['Label'] == label]
        
        if len(subset) < (test_per_class + val_per_class):
            raise ValueError(f"クラス {label} のデータ数が不足しています。")

        test_subset = subset.sample(n=test_per_class, random_state=RANDOM_STATE)
        subset_remaining = subset.drop(test_subset.index)
        
        val_subset = subset_remaining.sample(n=val_per_class, random_state=RANDOM_STATE)
        train_subset_remaining = subset_remaining.drop(val_subset.index)
        
        test_fixed = pd.concat([test_fixed, test_subset])
        val_fixed = pd.concat([val_fixed, val_subset])
        train_master = pd.concat([train_master, train_subset_remaining])
        
    test_fixed = test_fixed.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    val_fixed = val_fixed.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    train_master = train_master.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    print("--- 評価用データセット（固定枚数） ---")
    val_save_path = os.path.join(OUTPUT_DIR, 'val_fixed.csv')
    val_fixed.to_csv(val_save_path, index=False)
    print_distribution(val_fixed, "Fixed Validation Set")

    test_save_path = os.path.join(OUTPUT_DIR, 'test_fixed.csv')
    test_fixed.to_csv(test_save_path, index=False)
    print_distribution(test_fixed, "Fixed Test Set")
    print("-" * 30 + "\n")

    # 3. 学習用マスターからのサンプリング
    min_class_count = train_master['Label'].value_counts().min() # 少数派クラスの枚数 (例: 137)
    max_class_count = train_master['Label'].value_counts().max() # 多数派クラスの枚数 (例: 354)
    
    print("--- 学習用マスターデータ（残りの全データ） ---")
    print_distribution(train_master, "Train Master Set")
    print(f"※少数派クラスの枚数(全クラス重複なし可能): {min_class_count}枚")
    print(f"※多数派クラスの枚数(一般的なアップサンプリング目標): {max_class_count}枚")
    print("-" * 30 + "\n")

    # ★ 指定サイズに加えて「少数派に合わせるサイズ(137)」と「多数派に合わせるサイズ(354)」を自動追加
    target_sizes = sorted(list(set(TRAIN_SIZES + [min_class_count, max_class_count])))

    print(f"--- 学習用データセット（ALLOW_UPSAMPLING: {ALLOW_UPSAMPLING}） ---")
    for size in target_sizes:
        train_subset = pd.DataFrame()
        total_target_size = size * num_classes
        
        if ALLOW_UPSAMPLING:
            # 【True】クラスごとに指定枚数を確保（不足時は重複あり）
            for label in train_master['Label'].unique():
                subset = train_master[train_master['Label'] == label]
                
                # ★ ここで多数派(354)と少数派(137)の処理が自動で分かれます
                # クラス0 (354枚) -> size(354) <= len(354) なので「重複なし」で全件抽出
                # クラス1,2 (202, 137枚) -> size(354) > len なので「重複あり」で水増し抽出
                if size <= len(subset):
                    sampled = subset.sample(n=size, replace=False, random_state=RANDOM_STATE)
                else:
                    sampled = subset.sample(n=size, replace=True, random_state=RANDOM_STATE)
                train_subset = pd.concat([train_subset, sampled])
        else:
            # 【False】重複は許可せず、全体枚数をTrue時と同じに揃える（不均衡になる）
            if total_target_size > len(train_master):
                print(f"[スキップ] {size}/class (目標合計 {total_target_size}枚): 全データ数({len(train_master)}枚)を超えるため、重複なしでは生成不可能です。")
                continue
                
            if size <= min_class_count:
                for label in train_master['Label'].unique():
                    subset = train_master[train_master['Label'] == label]
                    sampled = subset.sample(n=size, replace=False, random_state=RANDOM_STATE)
                    train_subset = pd.concat([train_subset, sampled])
            else:
                train_subset = train_master.sample(n=total_target_size, replace=False, random_state=RANDOM_STATE)

        train_subset = train_subset.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
        
        # 出力ファイル名の決定
        if size == min_class_count:
            save_name = 'train_min_unique.csv'
            if ALLOW_UPSAMPLING:
                dataset_label = f"Train Set (Matched to Minority: {size}/class)"
            else:
                dataset_label = f"Train Set (Max Balanced, No Duplicates: {size}/class)"
        elif size == max_class_count:
            save_name = 'train_max_upsampled.csv'
            if ALLOW_UPSAMPLING:
                dataset_label = f"Train Set (Matched to Majority, Upsampled: {size}/class)" # ★求めていたデータ
            else:
                dataset_label = f"Train Set (Imbalanced, Total: {total_target_size})"
        else:
            save_name = f'train_{size}.csv'
            if size > min_class_count and ALLOW_UPSAMPLING:
                dataset_label = f"Train Set (Upsampled: {size}/class)"
            else:
                dataset_label = f"Train Set ({size}/class)"
             
        save_path = os.path.join(OUTPUT_DIR, save_name)
        train_subset.to_csv(save_path, index=False)
        
        print_distribution(train_subset, dataset_label)

if __name__ == "__main__":
    create_comparison_datasets()

Loading data...

--- 元データの分布 ---
[Original Valid Data] 合計: 836枚 (内訳 -> 0: 356枚, 1: 265枚, 2: 215枚)
------------------------------

--- 評価用データセット（固定枚数） ---
[Fixed Validation Set] 合計: 144枚 (内訳 -> 0: 48枚, 1: 48枚, 2: 48枚)
[Fixed Test Set] 合計: 144枚 (内訳 -> 0: 48枚, 1: 48枚, 2: 48枚)
------------------------------

--- 学習用マスターデータ（残りの全データ） ---
[Train Master Set] 合計: 548枚 (内訳 -> 0: 260枚, 1: 169枚, 2: 119枚)
※少数派クラスの枚数(全クラス重複なし可能): 119枚
※多数派クラスの枚数(一般的なアップサンプリング目標): 260枚
------------------------------

--- 学習用データセット（ALLOW_UPSAMPLING: True） ---
[Train Set (10/class)] 合計: 30枚 (内訳 -> 0: 10枚, 1: 10枚, 2: 10枚)
[Train Set (50/class)] 合計: 150枚 (内訳 -> 0: 50枚, 1: 50枚, 2: 50枚)
[Train Set (100/class)] 合計: 300枚 (内訳 -> 0: 100枚, 1: 100枚, 2: 100枚)
[Train Set (Matched to Minority: 119/class)] 合計: 357枚 (内訳 -> 0: 119枚, 1: 119枚, 2: 119枚)
[Train Set (Matched to Majority, Upsampled: 260/class)] 合計: 780枚 (内訳 -> 0: 260枚, 1: 260枚, 2: 260枚)


testにtrainのデータないか確認

In [17]:
import pandas as pd
import os
import glob

def verify_no_data_leakage():
    # --- 設定 ---
    pre = "/home/hiromu/energy"
    OUTPUT_DIR = f'{pre}/src/FF/AFF/dataset_upsamp_clean'
    
    test_csv = os.path.join(OUTPUT_DIR, 'test_fixed.csv')
    val_csv = os.path.join(OUTPUT_DIR, 'val_fixed.csv')
    
    # 1. 評価用データセットの読み込み
    try:
        test_df = pd.read_csv(test_csv)
        val_df = pd.read_csv(val_csv)
    except FileNotFoundError as e:
        print(f"エラー: 評価用ファイルが見つかりません ({e.filename})")
        print("先にデータセット作成スクリプトを実行してください。")
        return

    # ファイル名の集合（Set）を作成
    # Setを使うことで、重複判定（積集合）が非常に高速かつ正確になります
    test_filenames = set(test_df['filename'])
    val_filenames = set(val_df['filename'])
    
    # 2. trainファイル一覧を取得
    train_files = glob.glob(os.path.join(OUTPUT_DIR, 'train_*.csv'))
    
    if not train_files:
        print("エラー: 学習用(train)のCSVファイルが見つかりません。")
        return

    print(f"テストデータ ({len(test_filenames)}枚) および検証データ ({len(val_filenames)}枚) との重複をチェックします...\n")
    print("-" * 50)
    
    leakage_found = False
    
    # 3. 各trainファイルに対して重複確認
    for train_file in sorted(train_files):
        train_df = pd.read_csv(train_file)
        train_filenames = set(train_df['filename'])
        
        # 積集合（共通するファイル名）を抽出
        overlap_with_test = train_filenames.intersection(test_filenames)
        overlap_with_val = train_filenames.intersection(val_filenames)
        
        file_basename = os.path.basename(train_file)
        
        if overlap_with_test or overlap_with_val:
            leakage_found = True
            print(f"❌ [警告] {file_basename}: データリークを検出しました！")
            if overlap_with_test:
                print(f"   -> Testデータとの重複 ({len(overlap_with_test)}件): {list(overlap_with_test)[:3]} ...")
            if overlap_with_val:
                print(f"   -> Valデータとの重複 ({len(overlap_with_val)}件): {list(overlap_with_val)[:3]} ...")
        else:
            print(f"✅ {file_basename}: 正常 (重複 0件)")

    print("-" * 50)
    print("【診断結果】")
    if leakage_found:
        print("⚠️ データリーク（重複）が存在します。分割ロジックを見直すか、元データのCSVに同一ファイル名の別データが混ざっていないか確認してください。")
    else:
        print("🎉 すべてのTrainデータセットにおいて、Test/Valデータとの重複は一切ありませんでした！\n完全に隔離された状態ですので、安全に学習と評価を進められます。")

if __name__ == "__main__":
    verify_no_data_leakage()

テストデータ (144枚) および検証データ (144枚) との重複をチェックします...

--------------------------------------------------
✅ train_10.csv: 正常 (重複 0件)
✅ train_100.csv: 正常 (重複 0件)
✅ train_50.csv: 正常 (重複 0件)
✅ train_max_upsampled.csv: 正常 (重複 0件)
✅ train_min_unique.csv: 正常 (重複 0件)
--------------------------------------------------
【診断結果】
🎉 すべてのTrainデータセットにおいて、Test/Valデータとの重複は一切ありませんでした！
完全に隔離された状態ですので、安全に学習と評価を進められます。


比較検証用のデータセット
train:test = 8:2
trainが各画像10枚～190枚，最大

In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# --- 設定 ---
pre = "/home/hiromu/energy"
INPUT_CSV_PATH = f'{pre}/data/1201_humomentstest/feature_huall.csv'
IMG_ROOT = f'{pre}/data/1201_humomentstest'
OUTPUT_DIR = f'{pre}/src/FF/AFF/dataset_comparison' # 保存先

# ★ここで比較したい学習データ枚数（1クラスあたり）を指定
TRAIN_SIZES = [10,20,30,40,50,60,70,80,90, 100,110,120,130,140,150,160,170,180,190, 200, 300] 

# 評価データの割合設定 (train : val : test = 6 : 2 : 2)
TEST_RATIO = 0.2  # 全体の20%をTestに
VAL_RATIO_FROM_REMAINING = 0.25  # 残り80%のうちの25%（＝全体の20%）をValに
RANDOM_STATE = 42

def create_comparison_datasets():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # 1. データ読み込みと有効データ抽出
    print("Loading data...")
    raw_df = pd.read_csv(INPUT_CSV_PATH)
    valid_indices = []
    for i in range(len(raw_df)):
        img_path = os.path.join(IMG_ROOT, 'mask', raw_df.iloc[i]['category'], raw_df.iloc[i]['filename'])
        if os.path.exists(img_path):
            valid_indices.append(i)
    valid_df = raw_df.iloc[valid_indices].reset_index(drop=True)

    # 2. 全クラスを最小枚数に揃える（均衡化）
    counts = valid_df['Label'].value_counts()
    min_count = counts.min()
    print(f"Original Min Count: {min_count}")
    
    balanced_df = pd.DataFrame()
    for label in counts.index:
        subset = valid_df[valid_df['Label'] == label]
        balanced_df = pd.concat([balanced_df, subset.sample(n=min_count, random_state=RANDOM_STATE)])
    
    # 3. データを 6:2:2 に分割
    # 第1段階: 全体からTest (全体の20%) を分離し、残りを Train+Val (全体の80%) とする
    train_val_df, test_fixed = train_test_split(
        balanced_df, 
        test_size=TEST_RATIO, 
        stratify=balanced_df['Label'], 
        random_state=RANDOM_STATE
    )

    # 第2段階: Train+Val (80%) から Val (全体の20%相当) を分離し、残りを Train (全体の60%相当) とする
    train_master, val_fixed = train_test_split(
        train_val_df, 
        test_size=VAL_RATIO_FROM_REMAINING, 
        stratify=train_val_df['Label'], 
        random_state=RANDOM_STATE
    )
    
    # 固定のValidationデータとTestデータを保存
    val_save_path = os.path.join(OUTPUT_DIR, 'val_fixed.csv')
    val_fixed.to_csv(val_save_path, index=False)
    print(f"\n[Fixed Validation Set] Saved: {len(val_fixed)} samples -> {val_save_path}")

    test_save_path = os.path.join(OUTPUT_DIR, 'test_fixed.csv')
    test_fixed.to_csv(test_save_path, index=False)
    print(f"[Fixed Test Set]       Saved: {len(test_fixed)} samples -> {test_save_path}\n")

    # 4. 学習用マスター(60%)から指定枚数をサンプリングして保存
    max_train_per_class = train_master['Label'].value_counts().min()
    print(f"Max available training samples per class: {max_train_per_class}")

    # 指定されたサイズ + 最大サイズも含める
    target_sizes = sorted(list(set(TRAIN_SIZES + [max_train_per_class])))

    for size in target_sizes:
        if size > max_train_per_class:
            print(f"Skipping size {size} (Not enough data, max is {max_train_per_class})")
            continue
            
        # 各クラスから size 枚ずつ抽出
        train_subset = pd.DataFrame()
        for label in train_master['Label'].unique():
            subset = train_master[train_master['Label'] == label]
            train_subset = pd.concat([train_subset, subset.sample(n=size, random_state=RANDOM_STATE)])
        
        # シャッフルして保存
        train_subset = train_subset.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
        save_name = f'train_{size}.csv'
        if size == max_train_per_class:
             save_name = 'train_max.csv'
             
        save_path = os.path.join(OUTPUT_DIR, save_name)
        train_subset.to_csv(save_path, index=False)
        print(f"[Train Set {size}/class] Saved: {len(train_subset)} samples -> {save_path}")

if __name__ == "__main__":
    create_comparison_datasets()

Loading data...
Original Min Count: 240

[Fixed Validation Set] Saved: 144 samples -> /home/hiromu/energy/src/FF/AFF/dataset_comparison/val_fixed.csv
[Fixed Test Set]       Saved: 144 samples -> /home/hiromu/energy/src/FF/AFF/dataset_comparison/test_fixed.csv

Max available training samples per class: 144
[Train Set 10/class] Saved: 30 samples -> /home/hiromu/energy/src/FF/AFF/dataset_comparison/train_10.csv
[Train Set 20/class] Saved: 60 samples -> /home/hiromu/energy/src/FF/AFF/dataset_comparison/train_20.csv
[Train Set 30/class] Saved: 90 samples -> /home/hiromu/energy/src/FF/AFF/dataset_comparison/train_30.csv
[Train Set 40/class] Saved: 120 samples -> /home/hiromu/energy/src/FF/AFF/dataset_comparison/train_40.csv
[Train Set 50/class] Saved: 150 samples -> /home/hiromu/energy/src/FF/AFF/dataset_comparison/train_50.csv
[Train Set 60/class] Saved: 180 samples -> /home/hiromu/energy/src/FF/AFF/dataset_comparison/train_60.csv
[Train Set 70/class] Saved: 210 samples -> /home/hiromu/ener